In [ ]:
import os
!pip install openai
# !pip install langchain_core==0.3.72
# !pip install langchain==0.3.27
!pip install langchain_community
# !pip install bitsandbytes
!pip install datasets
!pip install langgraph
# !pip install langchain_google_genai
!pip install langchain_huggingface
# !pip install dppy
# !pip install llama-cpp-python

!pip install vllm[flashinfer]==0.20.2

import pickle

# Assuming 'example_data.pkl' is a pickle file containing a Python object
with open('/KD/context/tqa-rationales.pickle', 'rb') as file:
    loaded_object = pickle.load(file)
    print(loaded_object[0])
    print(len(loaded_object))
with open('/KD/context/tqa-reflections.pickle', 'rb') as file:
    loaded_reflection_object = pickle.load(file)
with open('/KD/context/tqa-rationales-indices-mapped.pickle', 'rb') as file:
    loaded_4_object = pickle.load(file)
!pip uninstall -y torchcodec

import os
allowed_llm_sources = ['google', 'gguf-llama-cpp', 'gguf-ctransformers' , 'transformers-pipeline']
llm_source = "transformers-pipeline"
# fill this part if llm has a gguf file
gguf_path = "models/gguf/llama-2-7b.Q4_K_M.gguf"
embedding_sources = ['google', 'huggingface']
embedding_source = embedding_sources[1]

import os
allowed_llm_sources = ['google', 'gguf-llama-cpp', 'gguf-ctransformers' , 'transformers-pipeline']
llm_source = "transformers-pipeline"
# fill this part if llm has a gguf file
gguf_path = "models/gguf/llama-2-7b.Q4_K_M.gguf"
embedding_sources = ['google', 'huggingface']
embedding_source = embedding_sources[1]

from accelerate import Accelerator
# from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_community.llms import CTransformers
from accelerate import Accelerator
from langchain_community.llms import LlamaCpp
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# from configuration import allowed_llm_sources, llm_source, gguf_path
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.llms import VLLM
from vllm.config import AttentionConfig
from vllm.v1.attention.backends.registry import AttentionBackendEnum

import os
# os.environ["VLLM_USE_V1"] = "0"
import langchain
# langchain.verbose = False
# langchain.llm_cache = False

llm = None
if llm_source == 'google':
    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash", google_api_key=os.environ['gemini_api_key']
    )
elif llm_source == 'gguf-ctransformers':
    accelerator = Accelerator()
    config = {'max_new_tokens': 512, 'repetition_penalty': 1.1,
              'context_length': 8000, 'temperature': 0, 'gpu_layers': 10, 'stream': True}
    llm = CTransformers(model=gguf_path,
                        model_type="llama", stream=True, config=config)
    llm, config = accelerator.prepare(llm, config)
elif llm_source == 'gguf-llama-cpp':
    # Change this value based on your model and your GPU VRAM pool.
    n_gpu_layers = 160
    # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_batch = 4096
    n_ctx = 4096

    callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

    # Make sure the model path is correct for your system!
    llm = LlamaCpp(
        model_path=gguf_path,
        n_gpu_layers=n_gpu_layers, n_batch=n_batch,
        callback_manager=callback_manager,
        temperature=0.2,
        max_tokens=2000,
        top_p=1,
        verbose=False,
        n_ctx=n_ctx,
        f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
        model_kwargs={
            "repetition_penalty": 5
        }
    )

elif llm_source == 'transformers-pipeline':
   import transformers
   import torch
   import os
   model_id = "selfrag/selfrag_llama2_7b"
   hf_token = os.environ['hf_token']
   os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"
   llm = VLLM(
    model="selfrag/selfrag_llama2_7b",
    max_num_seqs=1,
    trust_remote_code=True,  # mandatory for hf models
    max_new_tokens=100,
    top_p=1,
    temperature=0,
    tensor_parallel_size=2,
    attention_backend="TRITON_ATTN",
    # seed=42,
    dtype='half',
    vllm_kwargs={
        "seed": 42,
        "gpu_memory_utilization": 0.75,
        "attention_backend": "TRITON_ATTN",
        "max_num_seqs" : 1,
        # "enforce_eager" : True,
    },
   )
else :
  pass

import os
import re
from sklearn.utils import shuffle
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import Adam
from transformers import get_scheduler
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import spacy
from langchain_core.documents import Document
# from dppy.finite_dpps import FiniteDPP
import collections
import json
def lists_are_equal(list1,list2):
    if collections.Counter(list1) == collections.Counter(list2):
        return True
    else:
        return False

def find_most_frequent_list(lists):
    equal_counters = []
    for i in range(len(lists)):
        counter = 0
        for j in range(len(lists)):
            if lists_are_equal(lists[i],lists[j]):
                counter += 1
        equal_counters.append(counter);
    return lists[np.argmax(np.array(equal_counters))]

# Function to generate text
def generate_text(tokenizer, model, prompt, max_length=10000, temperature=0.1, top_p=0.9, skip_prompt=True):
    device = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_token_length = inputs["input_ids"].shape[-1]  # Get the number of tokens in the prompt
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode the output, optionally skipping the prompt
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if skip_prompt:
        # Convert tokens to text, starting after the prompt tokens
        generated_tokens = outputs[0][prompt_token_length:]
        decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return decoded_output

INCREASING_SIM = 0
class T5ProjNetRetrievalEvaluator():

    def __init__(self):
        self.seed = 42
        self.batch_size = 1
        self.num_epochs = 50
        self.LOW_LABEL = 0
        self.AMBIGOUS_LABEL = 1
        self.HIGH_LABEL = 2
        self.nlp = spacy.load('en_core_web_sm')
        self.nlp.max_length = 10000000
        self.main_index = 0
        # self.s_model= SentenceTransformer('sentence-transformers/all-roberta-large-v1').to('cpu')
        # self.s_model= SentenceTransformer('sentence-transformers/all-roberta-large-v1')
        self.s_model= SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-cos-v1')
        # self.s_model= SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')
        self.number_of_selected = {
            "highs": [],
            "mediums": [],
            "websearches": []
        }

    def load_pretrained_model(self, load_path='./outputs/ep8'):
        return self

    def evaluate_single(self, q: str, doc: str, i: int): 
        # Grab the original item from loaded_object
        orig_item = self.loaded_object[self.main_index][i]
        orig_label = orig_item[1]
        orig_doc = orig_item[2]
    
        def format_doc_for_return(d):
            """Wraps string in a Document, or returns the LangchainDoc as-is."""
            if isinstance(d, str):
                return Document(page_content=d, metadata={"source": "Knowledge"})
            return d
    
        def get_raw_text(d):
            """Extracts the string for equality comparison, whether it's a str or LangchainDoc."""
            return d if isinstance(d, str) else d.page_content
        # ------------------------
    
        # If it is NOT AMBIGUOUS, return it like we do now
        if orig_label != self.AMBIGOUS_LABEL:
            return False, orig_label, format_doc_for_return(orig_doc)
    
        # It IS AMBIGUOUS. Find the matching document in loaded_4_object
        target_text = get_raw_text(orig_doc)
        found_index = -1
        
        # Iterate through loaded_4_object to find the real index
        for idx, item in enumerate(self.loaded_4_object[self.main_index]):
            if get_raw_text(item[2]) == target_text:
                found_index = idx
                break
    
        # If the index IS found, get the result from loaded_reflection_object
        if found_index != -1:
            ref_item = self.loaded_reflection_object[self.main_index][found_index]
            ref_label = ref_item[1]
            ref_doc = ref_item[2]
            return True, ref_label, format_doc_for_return(ref_doc)
            
        return True, orig_label, format_doc_for_return(orig_doc)
    def evaluate_batch(self, q: str, docs: list):
        # Initialize lists for different categories
        high_docs = []
        all_docs = []
        # Sort strips using sort_docs
        # Create original indices to track them during sorting
        original_indices = list(range(len(docs)))
        
        # Sort documents and their indices, setting use_reasons=True
        sorted_docs, sorted_indices = self.sort_docs(q, docs, original_indices, use_reasons=True)
        reserved_highs = []
        # Evaluate sorted strips and categorize, break if high category reaches 3
        for doc, orig_i in zip(sorted_docs, sorted_indices):
            reflection, out, text = self.evaluate_single(q, doc, orig_i)
            all_docs.append(text)
            if out != self.LOW_LABEL:
              high_docs.append(text)
            if len(high_docs) >= 5:
              break
        self.main_index += 1
        if len(high_docs) > 0:  # At least one strip is labeled "high"
            self.number_of_selected["highs"].append(len(high_docs) if len(high_docs) < 3 else 3)
            self.number_of_selected["mediums"].append(0)
            self.number_of_selected["websearches"].append(0)
            return high_docs[:5], "No"
        else:
            self.number_of_selected["highs"].append(0)
            self.number_of_selected["mediums"].append(0)
            self.number_of_selected["websearches"].append(0)
            return all_docs[:5], "No"
    def sort_docs(self, q, docs, indices, use_reasons=False):
        s_model = self.s_model
        q_emb = s_model.encode(q)
        scores = []
        INCREASING_SIM = 1000.1
        # Iterate over docs and their original indices at the same time
        for idx, doc in zip(indices, docs):
            
            # Decide what we are embedding (the reason from memory, or the doc itself)
            if use_reasons:
                # Extract the reasoning using the original integer index (idx)
                item_to_embed = self.loaded_object[self.main_index][idx][2]
            else:
                item_to_embed = doc
                
            # Handle whether it is a string or a Document object
            if isinstance(item_to_embed, str):
                text_to_encode = item_to_embed
            else:
                text_to_encode = item_to_embed.page_content
                
            # Encode and score
            scores.append(INCREASING_SIM)
            INCREASING_SIM -= 0.1

        avg_scores = [score for score in scores]
        top_n_indices = np.argsort(np.array(avg_scores))[::-1]
        top_corresponding_values = [docs[i] for i in top_n_indices]
        top_corresponding_indices = [indices[i] for i in top_n_indices]
        
        return top_corresponding_values, top_corresponding_indices
    def evaluate_websearch(self, q: str, docs: list, num_of_medium_docs=0,i=0):
          new_docs, _ = self.sort_docs(q,docs,docs)
          new_docs = docs[:5]
          high_docs = []
          high_probs = []
          for d in new_docs:
          # # for d in strips:
              prob, out, text = self.evaluate_single(q, str(d))
              if out == self.HIGH_LABEL:
                  high_docs.append(str(text))
                  high_probs.append(prob)
          self.number_of_selected["websearches"].append(len(high_docs) if len(high_docs) < 3 else 3)
          return high_docs[:5 if num_of_medium_docs == 0 else 3], "No"


from src.DataLoaders.DataLoader import DataLoader
# from src.DataLoaders.Arc import Arc
# from src.DataLoaders.PubHealth import PubHealth
# from src.DataLoaders.PopQA import PopQA
from src.DataLoaders.TQAUnfiltered import TQAUnfiltered
from src.RetrievalEvaluators.RetrievalEvaluator import RetrievalEvaluator
from graph import workflow_compiler
app = workflow_compiler()
# dataloader, retrieval_evaluator= Arc(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for ARC
# dataloader, retrieval_evaluator= PubHealth(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for Pubhealth
# dataloader, retrieval_evaluator= PopQA(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for PopQA
dataloader, retrieval_evaluator= TQAUnfiltered(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for TQA
# from models.LLM import llm

retrieval_evaluator.loaded_object= loaded_object
retrieval_evaluator.loaded_4_object = loaded_4_object
retrieval_evaluator.loaded_reflection_object= loaded_reflection_object

def generate_rag_response(input_text, index):
    ans = ""
    input_dict = {"question": str(input_text) , "dataloader" : dataloader , "retrieval_evaluator" : retrieval_evaluator, 'llm': llm, 'index': index}
    # print(input_dict)
    response = app.invoke(input_dict)
    ans = response['generation']
    # print(ans)
    return ans

# Run this and set load_generations=True to start from checkpoint
load_generations = False
checkpoint = 925
# check the filename to match testing dataset
pickle_file_name = '/outputs/tqa_temp_925.pickle'
import pickle
if load_generations == True:
  with open(pickle_file_name, 'rb') as handle:
    res = pickle.load(handle)
    dataloader.generations = res['generations'][:checkpoint]
    dataloader.generation_checkpoint = checkpoint
from src.Helpers.Process import Process
with torch.no_grad():
  process = Process(dataloader)
  process.start(generate_rag_response,load_sample_data_only=False,template='self-rag')
print("The final accuracy is: ",process.accuracy)

dataloader.accuracy()

def save_model_outputs(self, save_path: str = '/outputs/tqa-final-newprompt-3.pickle') -> bool:
     with open(save_path, 'wb') as handle:
         pickle.dump({'input_test_data': self.input_test_data, 'generations': self.generations}, handle)
import pickle


save_model_outputs(dataloader)